# Lesson 9B: Convolutional Neural Network Practical

<a name="introduction"></a>
## Introduction

Lesson 9a derived convolution, backpropagation through conv and pooling
layers, and weight sharing, then verified a from-scratch NumPy implementation
by numerical gradient checking and trained it on a small MNIST subset. This
lesson moves to a harder, more realistic image classification problem
(CIFAR-10: 32x32 color photographs across 10 classes, much less separable
than handwritten digits) and to production tooling: PyTorch for training a
CNN from scratch, and transfer learning from an ImageNet-pretrained ResNet.

In this lesson, we'll:
1. Design and train a small CNN from scratch in PyTorch on CIFAR-10
2. Derive the mathematics of transfer learning: feature reuse, frozen layers, fine-tuning
3. Apply a pretrained ResNet-18, freeze its backbone, and fine-tune only the classification head
4. Visualize learned filters and activation maps to build intuition for what early and late layers detect
5. Re-run 9a's from-scratch NumPy CNN on a CIFAR-10 subset and compare it directly to the PyTorch implementation


## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Dataset: CIFAR-10](#dataset-cifar-10)
4. [CNN Architecture Design](#cnn-architecture-design)
5. [Training From Scratch with PyTorch](#training-from-scratch-with-pytorch)
6. [Transfer Learning Mathematics](#transfer-learning-mathematics)
   - [Feature Reuse](#feature-reuse)
   - [Frozen Layers and Fine-Tuning](#frozen-layers-and-fine-tuning)
7. [Applying a Pretrained ResNet-18](#applying-a-pretrained-resnet-18)
8. [Feature and Filter Visualization](#feature-and-filter-visualization)
9. [From-Scratch NumPy CNN on CIFAR-10](#from-scratch-numpy-cnn-on-cifar-10)
   - [NumPy vs PyTorch Comparison](#numpy-vs-pytorch-comparison)
10. [Performance Analysis](#performance-analysis)
11. [Conclusion](#conclusion)
    - [Key Insights](#key-insights-8)
    - [Further Reading](#further-reading-8)


<a name="required-libraries"></a>
## Required Libraries

In [ ]:
# Colab preinstalls numpy/matplotlib/torch/torchvision/scikit-learn.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
torch.manual_seed(42)
np.random.seed(42)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']


<a name="dataset-cifar-10"></a>
## Dataset: CIFAR-10

CIFAR-10 is a substantially harder classification problem than MNIST: 10
classes of real-world 32x32 color photographs (airplanes, cats, ships, etc.),
with far more intra-class variation and far less separability than
handwritten digits. We use a 20,000-image subset (the full CIFAR-10 has
60,000 images) for tractable training time without a GPU.


In [ ]:
cifar = fetch_openml('CIFAR_10_small', version=1, as_frame=False, parser='auto')
X_all = cifar.data.reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
y_all = cifar.target.astype(int)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
# Further subsample the training set for CPU-tractable training time
n_train = 4000
rng = np.random.RandomState(42)
train_subset_idx = rng.choice(len(X_train_full), n_train, replace=False)
X_train = X_train_full[train_subset_idx]
y_train = y_train_full[train_subset_idx]

print("\n" + "="*70)
print("CIFAR-10 DATASET")
print("="*70)
print(f"\nTraining images (subset used): {X_train.shape[0]}")
print(f"Test images: {X_test.shape[0]}")
print(f"Image shape: {X_train.shape[1:]}")
print(f"Classes: {CLASS_NAMES}")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, label in zip(axes.flat, X_train[:16], y_train[:16]):
    ax.imshow(np.transpose(img, (1, 2, 0)))
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample CIFAR-10 Images')
plt.tight_layout()
plt.show()


<a name="cnn-architecture-design"></a>
## CNN Architecture Design

Design choices for a small CIFAR-10 CNN, and why:

- **Kernel size 3x3**: the standard choice since VGG (2014) — stacking small
  filters grows the receptive field (derived in 9a) at far fewer parameters
  than one large filter, and adds more nonlinearities per unit of receptive
  field growth
- **Increasing filter counts (16 -> 32 -> 64)**: as spatial resolution shrinks
  through pooling, increasing the channel count preserves representational
  capacity — early layers need few filters to detect simple features (edges,
  color blobs), later layers need many to represent combinations of those
  features
- **Max pooling after each conv block**: halves spatial resolution, reduces
  computation for subsequent layers, and provides a small amount of
  translation invariance (derived in 9a)
- **A final fully-connected classifier**: after enough conv+pool blocks that
  the spatial resolution is small, a small number of FC layers combine the
  learned spatial features into class scores


In [ ]:
class CIFAR_CNN(nn.Module):
    """
    Conv(16,3x3)-ReLU-Pool -> Conv(32,3x3)-ReLU-Pool -> Conv(64,3x3)-ReLU-Pool
    -> Flatten -> Dense(128) -> Dense(10)
    """
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        # 32x32 -> pool -> 16x16 -> pool -> 8x8 -> pool -> 4x4
        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, n_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.reshape(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(x)

model = CIFAR_CNN()
n_params = sum(p.numel() for p in model.parameters())
print("\n" + "="*70)
print("CIFAR CNN ARCHITECTURE")
print("="*70)
print(model)
print(f"\nTotal trainable parameters: {n_params:,}")


<a name="training-from-scratch-with-pytorch"></a>
## Training From Scratch with PyTorch

In [ ]:
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

n_epochs = 15
batch_size = 64
n_train = len(X_train_t)

train_losses, train_accs, test_accs = [], [], []

t0 = time.time()
for epoch in range(n_epochs):
    model.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    correct = 0

    for start in range(0, n_train, batch_size):
        idx = perm[start:start + batch_size]
        xb, yb = X_train_t[idx], y_train_t[idx]

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(idx)
        correct += (logits.argmax(dim=1) == yb).sum().item()

    train_losses.append(epoch_loss / n_train)
    train_accs.append(correct / n_train)

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_acc = (test_logits.argmax(dim=1) == y_test_t).float().mean().item()
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1}/{n_epochs}: loss={train_losses[-1]:.4f}, "
          f"train_acc={train_accs[-1]:.4f}, test_acc={test_acc:.4f}")

pytorch_scratch_time = time.time() - t0
print(f"\nTotal training time: {pytorch_scratch_time:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, n_epochs+1), train_losses, linewidth=2, color='darkred')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training loss')
axes[0].set_title('PyTorch CNN: Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, n_epochs+1), train_accs, linewidth=2, label='Train accuracy')
axes[1].plot(range(1, n_epochs+1), test_accs, linewidth=2, label='Test accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('PyTorch CNN: Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

pytorch_scratch_acc = test_accs[-1]
print(f"\nFinal test accuracy: {pytorch_scratch_acc:.4f}")


<a name="transfer-learning-mathematics"></a>
## Transfer Learning Mathematics

<a name="feature-reuse"></a>
### Feature Reuse

A CNN trained on a large, diverse dataset (ImageNet: 1.2 million images, 1000
classes) learns a hierarchy of features: early layers learn generic,
task-independent patterns (oriented edges, color blobs, simple textures) that
are useful for essentially *any* natural image classification task; later
layers learn increasingly task-specific combinations of those features (e.g.
"dog snout" or "car wheel" detectors).

**Transfer learning** exploits this hierarchy: rather than training a new
network from random initialization, we start from the pretrained weights and
reuse the early, generic layers directly — only the later, task-specific
layers need to be adapted, and often only the final classification layer.

<a name="frozen-layers-and-fine-tuning"></a>
### Frozen Layers and Fine-Tuning

Mathematically, "freezing" a layer means setting its gradient contribution to
zero during the parameter update: for a frozen layer with parameters
$\theta_{\text{frozen}}$,

$$\theta_{\text{frozen}}^{(t+1)} = \theta_{\text{frozen}}^{(t)} \quad \text{(no update, regardless of } \nabla_\theta L \text{)}$$

while unfrozen (fine-tuned) parameters update normally:

$$\theta_{\text{trainable}}^{(t+1)} = \theta_{\text{trainable}}^{(t)} - \eta \nabla_{\theta_{\text{trainable}}} L$$

Gradients still flow *backward through* frozen layers during backpropagation
(the chain rule doesn't stop at a frozen layer — the trainable layers behind
it still need $\partial L/\partial x$ passed through), but the frozen
layer's own weights simply never change. This makes fine-tuning **far
cheaper** than full training: with $P_{\text{total}}$ total parameters and
only $P_{\text{trainable}} \ll P_{\text{total}}$ trainable (e.g. just the
final linear layer), the optimizer's parameter-update cost — and, in the
common case of Adam, its per-parameter moment estimates — scale with
$P_{\text{trainable}}$, not $P_{\text{total}}$.

A common strategy: freeze everything, train only the new head for a few
epochs (fast, avoids destroying the pretrained features early on with large,
poorly-calibrated gradients from a freshly-initialized head), then optionally
unfreeze the last few layers and fine-tune the whole network at a much
smaller learning rate.


In [ ]:
print("\n" + "="*70)
print("TRANSFER LEARNING: PARAMETER COST COMPARISON")
print("="*70)
resnet_probe = models.resnet18(weights=None)
total_params = sum(p.numel() for p in resnet_probe.parameters())
# resnet_probe still has its original 1000-class ImageNet head; compute what a
# 10-class head (the one actually used below for CIFAR-10) would cost instead
# of measuring the mismatched 1000-class head.
head_in_features = resnet_probe.fc.in_features
ten_class_head_params = head_in_features * 10 + 10
print(f"\nResNet-18 total parameters: {total_params:,}")
print(f"10-class head parameters ({head_in_features} -> 10): {ten_class_head_params:,}")
print(f"Fraction trainable when only the head is fine-tuned: {ten_class_head_params/total_params:.4%}")
print("\nTraining just the head means the optimizer only tracks gradients and")
print("moment estimates for well under 0.1% of the network's parameters.")


<a name="applying-a-pretrained-resnet-18"></a>
## Applying a Pretrained ResNet-18

In [ ]:
# Load ImageNet-pretrained ResNet-18, freeze the backbone, replace the head
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in resnet.parameters():
    param.requires_grad = False  # freeze backbone

resnet.fc = nn.Linear(resnet.fc.in_features, 10)  # new, trainable head

# ResNet-18 expects ImageNet-style normalized 224x224 input
imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def preprocess_for_resnet(x_batch):
    x_resized = torch.nn.functional.interpolate(x_batch, size=224, mode='bilinear', align_corners=False)
    return (x_resized - imagenet_mean) / imagenet_std

# Use a smaller subset for the transfer-learning fine-tune loop (224x224 is
# far more expensive per image on CPU than the 32x32 from-scratch training above)
n_transfer = 1200
transfer_idx = rng.choice(len(X_train_full), n_transfer, replace=False)
X_transfer = torch.from_numpy(X_train_full[transfer_idx])
y_transfer = torch.from_numpy(y_train_full[transfer_idx])

n_test_transfer = 400
test_transfer_idx = rng.choice(len(X_test), n_test_transfer, replace=False)
X_test_transfer = torch.from_numpy(X_test[test_transfer_idx])
y_test_transfer = torch.from_numpy(y_test[test_transfer_idx])

print(f"\nFine-tuning on {n_transfer} images, evaluating on {n_test_transfer} "
      f"(224x224 resize makes this far more expensive per image than the "
      f"32x32 from-scratch CNN above, hence the smaller subset).")


In [ ]:
optimizer_ft = torch.optim.Adam(resnet.fc.parameters(), lr=0.001)  # only the head has requires_grad=True
criterion_ft = nn.CrossEntropyLoss()

n_epochs_ft = 5
batch_size_ft = 32

ft_losses, ft_test_accs = [], []

t0 = time.time()
for epoch in range(n_epochs_ft):
    resnet.train()
    perm = torch.randperm(n_transfer)
    epoch_loss = 0.0

    for start in range(0, n_transfer, batch_size_ft):
        idx = perm[start:start + batch_size_ft]
        xb = preprocess_for_resnet(X_transfer[idx])
        yb = y_transfer[idx]

        optimizer_ft.zero_grad()
        logits = resnet(xb)
        loss = criterion_ft(logits, yb)
        loss.backward()
        optimizer_ft.step()

        epoch_loss += loss.item() * len(idx)

    ft_losses.append(epoch_loss / n_transfer)

    resnet.eval()
    with torch.no_grad():
        test_logits = resnet(preprocess_for_resnet(X_test_transfer))
        test_acc = (test_logits.argmax(dim=1) == y_test_transfer).float().mean().item()
    ft_test_accs.append(test_acc)

    print(f"Epoch {epoch+1}/{n_epochs_ft}: loss={ft_losses[-1]:.4f}, test_acc={test_acc:.4f}")

transfer_time = time.time() - t0
print(f"\nTransfer learning time: {transfer_time:.1f}s (only {sum(p.numel() for p in resnet.fc.parameters()):,} "
      f"parameters trained)")

fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.plot(range(1, n_epochs_ft+1), ft_test_accs, 'o-', linewidth=2, color='seagreen')
ax.set_xlabel('Epoch')
ax.set_ylabel('Test accuracy')
ax.set_title('Transfer Learning: Frozen ResNet-18 Backbone, Fine-Tuned Head')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

resnet_transfer_acc = ft_test_accs[-1]
print(f"\nFinal transfer-learning test accuracy: {resnet_transfer_acc:.4f} "
      f"(on {n_transfer} training images, vs {n_train} for the from-scratch CNN above)")


<a name="feature-and-filter-visualization"></a>
## Feature and Filter Visualization

In [ ]:
# Visualize ResNet-18's first-layer filters -- classic result: early layers
# learn oriented edge detectors and color-opponent blobs, not object parts
first_layer_weights = resnet.conv1.weight.data.clone()  # (64, 3, 7, 7)

# Normalize each filter to [0,1] for display
def normalize_filter(f):
    f = f - f.min()
    return f / (f.max() + 1e-8)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for idx, ax in enumerate(axes.flat):
    filt = normalize_filter(first_layer_weights[idx]).permute(1, 2, 0).numpy()
    ax.imshow(filt)
    ax.axis('off')
plt.suptitle("ResNet-18's First-Layer Filters (pretrained on ImageNet)")
plt.tight_layout()
plt.show()

print("\nMost filters show oriented edges or color-opponent blobs (e.g. a")
print("green-magenta or blue-yellow gradient) -- exactly the kind of generic,")
print("task-independent features the transfer-learning argument above")
print("depends on. None of these filters look like 'cat ear' or 'car wheel'")
print("detectors -- those specific detectors emerge in later layers, closer")
print("to the classification head.")


In [ ]:
# Visualize activation maps from the from-scratch PyTorch CNN's first conv layer
model.eval()
sample_img = X_test_t[:1]
with torch.no_grad():
    activations = model.relu(model.conv1(sample_img))  # (1, 16, 32, 32)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(np.transpose(sample_img[0].numpy(), (1, 2, 0)))
axes[0].set_title(f'Input ({CLASS_NAMES[y_test[0]]})')
axes[0].axis('off')

for i, ax in enumerate(axes[1:]):
    ax.imshow(activations[0, i].numpy(), cmap='viridis')
    ax.set_title(f'Filter {i}')
    ax.axis('off')

plt.suptitle("From-Scratch CNN's First-Layer Activation Maps")
plt.tight_layout()
plt.show()

print("\nEach activation map highlights where in the image that particular")
print("filter's learned pattern is present -- some filters respond to edges")
print("at specific orientations, others to broad color regions.")


<a name="from-scratch-numpy-cnn-on-cifar-10"></a>
## From-Scratch NumPy CNN on CIFAR-10

We reuse the exact `Conv2D` and `MaxPool2D` implementations from Lesson 9a
(verified there by numerical gradient checking) and apply them to a CIFAR-10
subset, to confirm the from-scratch implementation generalizes beyond the
grayscale MNIST digits it was originally demonstrated on.


In [ ]:
class Conv2D:
    """From Lesson 9a -- reproduced here since 9a's implementation is not
    importable as a package; identical code, already gradient-checked there."""

    def __init__(self, n_filters, filter_size, in_channels, stride=1, padding=0):
        self.n_filters, self.filter_size, self.in_channels = n_filters, filter_size, in_channels
        self.stride, self.padding = stride, padding
        scale = np.sqrt(2.0 / (filter_size * filter_size * in_channels))
        self.W = np.random.randn(n_filters, in_channels, filter_size, filter_size) * scale
        self.b = np.zeros(n_filters)

    def _pad(self, x):
        if self.padding == 0:
            return x
        return np.pad(x, ((0, 0), (0, 0), (self.padding, self.padding), (self.padding, self.padding)))

    def forward(self, x):
        self.x = x
        F = self.filter_size
        xp = self._pad(x)
        self.x_padded = xp
        Hp, Wp = xp.shape[2], xp.shape[3]
        out_h = (Hp - F) // self.stride + 1
        out_w = (Wp - F) // self.stride + 1
        out = np.zeros((x.shape[0], self.n_filters, out_h, out_w))
        for i in range(out_h):
            for j in range(out_w):
                hs, ws = i * self.stride, j * self.stride
                patch = xp[:, :, hs:hs + F, ws:ws + F]
                out[:, :, i, j] = np.tensordot(patch, self.W, axes=([1, 2, 3], [1, 2, 3])) + self.b
        self.out_shape = (out_h, out_w)
        return out

    def backward(self, dout):
        F = self.filter_size
        xp = self.x_padded
        out_h, out_w = self.out_shape
        dW = np.zeros_like(self.W)
        db = np.sum(dout, axis=(0, 2, 3))
        dxp = np.zeros_like(xp)
        for i in range(out_h):
            for j in range(out_w):
                hs, ws = i * self.stride, j * self.stride
                patch = xp[:, :, hs:hs + F, ws:ws + F]
                dW += np.tensordot(dout[:, :, i, j], patch, axes=([0], [0]))
                dxp[:, :, hs:hs + F, ws:ws + F] += np.tensordot(dout[:, :, i, j], self.W, axes=([1], [0]))
        dx = dxp[:, :, self.padding:dxp.shape[2]-self.padding, self.padding:dxp.shape[3]-self.padding] if self.padding > 0 else dxp
        self.dW, self.db = dW, db
        return dx


class MaxPool2D:
    def __init__(self, size=2, stride=2):
        self.size, self.stride = size, stride

    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        s = self.size
        out_h = (H - s) // self.stride + 1
        out_w = (W - s) // self.stride + 1
        out = np.zeros((N, C, out_h, out_w))
        self.argmax = np.zeros((N, C, out_h, out_w, 2), dtype=int)
        for i in range(out_h):
            for j in range(out_w):
                hs, ws = i * self.stride, j * self.stride
                patch = x[:, :, hs:hs+s, ws:ws+s]
                flat = patch.reshape(N, C, -1)
                idx = np.argmax(flat, axis=2)
                out[:, :, i, j] = np.max(flat, axis=2)
                self.argmax[:, :, i, j, 0] = idx // s
                self.argmax[:, :, i, j, 1] = idx % s
        self.out_shape = (out_h, out_w)
        return out

    def backward(self, dout):
        N, C, H, W = self.x.shape
        s = self.size
        out_h, out_w = self.out_shape
        dx = np.zeros_like(self.x)
        for i in range(out_h):
            for j in range(out_w):
                hs, ws = i * self.stride, j * self.stride
                for n in range(N):
                    for c in range(C):
                        di, dj = self.argmax[n, c, i, j]
                        dx[n, c, hs+di, ws+dj] += dout[n, c, i, j]
        return dx


class Dense:
    def __init__(self, in_features, out_features):
        scale = np.sqrt(2.0 / in_features)
        self.W = np.random.randn(in_features, out_features) * scale
        self.b = np.zeros(out_features)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout
        self.db = np.sum(dout, axis=0)
        return dout @ self.W.T


def relu_forward(x):
    return np.maximum(0, x)


def relu_backward(dout, x):
    return dout * (x > 0)


def softmax_cross_entropy(logits, y_true):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    n = logits.shape[0]
    loss = np.mean(-np.log(probs[np.arange(n), y_true] + 1e-12))
    dlogits = probs.copy()
    dlogits[np.arange(n), y_true] -= 1
    dlogits /= n
    return loss, dlogits


class NumpyCIFARCNN:
    """Conv(8,3x3) -> ReLU -> MaxPool(2x2) -> Flatten -> Dense(10)."""

    def __init__(self, n_classes=10):
        self.conv = Conv2D(n_filters=8, filter_size=3, in_channels=3, stride=1, padding=1)
        self.pool = MaxPool2D(size=2, stride=2)
        self.dense = Dense(in_features=8 * 16 * 16, out_features=n_classes)

    def forward(self, x):
        self.conv_out = self.conv.forward(x)
        self.relu_out = relu_forward(self.conv_out)
        self.pool_out = self.pool.forward(self.relu_out)
        self.flat_shape = self.pool_out.shape
        flat = self.pool_out.reshape(self.pool_out.shape[0], -1)
        return self.dense.forward(flat)

    def backward(self, dlogits):
        dflat = self.dense.backward(dlogits)
        dpool_out = dflat.reshape(self.flat_shape)
        drelu_out = self.pool.backward(dpool_out)
        dconv_out = relu_backward(drelu_out, self.conv_out)
        self.conv.backward(dconv_out)

    def step(self, lr):
        self.conv.W -= lr * self.conv.dW
        self.conv.b -= lr * self.conv.db
        self.dense.W -= lr * self.dense.dW
        self.dense.b -= lr * self.dense.db

    def predict(self, x, batch_size=64):
        preds = []
        for i in range(0, len(x), batch_size):
            preds.append(np.argmax(self.forward(x[i:i+batch_size]), axis=1))
        return np.concatenate(preds)


print("\n" + "="*70)
print("NUMPY CIFAR CNN: Conv(8,3x3) -> ReLU -> MaxPool -> Dense(10)")
print("="*70)
print("\nUsing 9a's verified Conv2D/MaxPool2D, applied directly to 3-channel")
print("CIFAR-10 images (in_channels=3) rather than 9a's single-channel MNIST.")


In [ ]:
# Train the NumPy CNN on a small CIFAR-10 subset (kept small: pure-Python
# nested loops over spatial positions make this much slower than PyTorch's
# vectorized/compiled convolution)
n_numpy_train, n_numpy_test = 500, 150
numpy_train_idx = rng.choice(len(X_train_full), n_numpy_train, replace=False)
numpy_test_idx = rng.choice(len(X_test), n_numpy_test, replace=False)

X_np_train = X_train_full[numpy_train_idx]
y_np_train = y_train_full[numpy_train_idx]
X_np_test = X_test[numpy_test_idx]
y_np_test = y_test[numpy_test_idx]

numpy_model = NumpyCIFARCNN(n_classes=10)
n_epochs_np = 6
batch_size_np = 32
lr_np = 0.05

numpy_losses, numpy_test_accs = [], []

t0 = time.time()
for epoch in range(n_epochs_np):
    perm = rng.permutation(n_numpy_train)
    epoch_losses = []
    for start in range(0, n_numpy_train, batch_size_np):
        idx = perm[start:start + batch_size_np]
        xb, yb = X_np_train[idx], y_np_train[idx]
        logits = numpy_model.forward(xb)
        loss, dlogits = softmax_cross_entropy(logits, yb)
        numpy_model.backward(dlogits)
        numpy_model.step(lr_np)
        epoch_losses.append(loss)

    test_pred = numpy_model.predict(X_np_test)
    test_acc = accuracy_score(y_np_test, test_pred)
    numpy_losses.append(np.mean(epoch_losses))
    numpy_test_accs.append(test_acc)
    print(f"Epoch {epoch+1}/{n_epochs_np}: loss={numpy_losses[-1]:.4f}, test_acc={test_acc:.4f}")

numpy_time = time.time() - t0
print(f"\nNumPy CNN training time: {numpy_time:.1f}s "
      f"({n_numpy_train} train / {n_numpy_test} test images)")


<a name="numpy-vs-pytorch-comparison"></a>
### NumPy vs PyTorch Comparison

In [ ]:
# Train an equivalent-scale PyTorch CNN on the SAME small subset for a fair
# runtime and accuracy comparison
class TinyPyTorchCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(8 * 16 * 16, n_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv(x)))
        x = x.reshape(x.size(0), -1)
        return self.fc(x)

torch.manual_seed(42)
tiny_model = TinyPyTorchCNN()
tiny_opt = torch.optim.SGD(tiny_model.parameters(), lr=lr_np)
tiny_crit = nn.CrossEntropyLoss()

X_np_train_t = torch.from_numpy(X_np_train)
y_np_train_t = torch.from_numpy(y_np_train)
X_np_test_t = torch.from_numpy(X_np_test)
y_np_test_t = torch.from_numpy(y_np_test)

t0 = time.time()
for epoch in range(n_epochs_np):
    perm = torch.randperm(n_numpy_train)
    for start in range(0, n_numpy_train, batch_size_np):
        idx = perm[start:start + batch_size_np]
        xb, yb = X_np_train_t[idx], y_np_train_t[idx]
        tiny_opt.zero_grad()
        loss = tiny_crit(tiny_model(xb), yb)
        loss.backward()
        tiny_opt.step()

pytorch_tiny_time = time.time() - t0
with torch.no_grad():
    tiny_pytorch_acc = (tiny_model(X_np_test_t).argmax(dim=1) == y_np_test_t).float().mean().item()

print("\n" + "="*70)
print("NUMPY vs PYTORCH: SAME ARCHITECTURE, SAME DATA, SAME EPOCHS")
print("="*70)
print(f"\n{'Implementation':<20}{'Test Accuracy':<18}{'Training Time (s)':<20}")
print("-"*60)
print(f"{'NumPy (from-scratch)':<20}{numpy_test_accs[-1]:<18.4f}{numpy_time:<20.2f}")
print(f"{'PyTorch':<20}{tiny_pytorch_acc:<18.4f}{pytorch_tiny_time:<20.2f}")
print(f"\nPyTorch is {numpy_time/pytorch_tiny_time:.1f}x faster than the from-scratch")
print("NumPy implementation on identical data and architecture -- the speedup")
print("comes from PyTorch's compiled/vectorized convolution kernels (and")
print("optional GPU offload), not from a different algorithm. Both compute")
print("the same mathematical operation derived in Lesson 9a.")


<a name="performance-analysis"></a>
## Performance Analysis

In [ ]:
# Full performance analysis for the main from-scratch PyTorch CNN (trained
# on the larger 4000-image subset earlier)
model.eval()
with torch.no_grad():
    final_test_logits = model(X_test_t)
    final_test_pred = final_test_logits.argmax(dim=1).numpy()

cm = confusion_matrix(y_test, final_test_pred)

fig, ax = plt.subplots(1, 1, figsize=(9, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Confusion Matrix: PyTorch CNN (Test Set)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, final_test_pred, target_names=CLASS_NAMES))


In [ ]:
# Show a handful of misclassified examples
misclassified = np.where(final_test_pred != y_test)[0]
print(f"\n{len(misclassified)} / {len(y_test)} test images misclassified "
      f"({len(misclassified)/len(y_test):.1%})")

n_show = 8
show_idx = misclassified[:n_show]
fig, axes = plt.subplots(1, n_show, figsize=(16, 3))
for ax, i in zip(axes, show_idx):
    img = np.transpose(X_test[i], (1, 2, 0))
    ax.imshow(img)
    ax.set_title(f"True: {CLASS_NAMES[y_test[i]]}\nPred: {CLASS_NAMES[final_test_pred[i]]}", fontsize=8)
    ax.axis('off')
plt.suptitle('Misclassified Examples')
plt.tight_layout()
plt.show()

print("\nMisclassifications concentrate on visually similar classes (e.g. cat")
print("vs dog, automobile vs truck) -- consistent with CIFAR-10 being a")
print("genuinely harder problem than MNIST, where classes are far more")
print("visually distinct.")


<a name="conclusion"></a>
## Conclusion

<a name="key-insights-8"></a>
### Key Insights

1. **Architecture design choices** (small filters, increasing channel counts,
   pooling between conv blocks) balance receptive field growth against
   parameter and computation cost

2. **Transfer learning reuses generic early-layer features** learned on a
   large source dataset (ImageNet); freezing those layers and training only
   a new head is both mathematically justified (feature hierarchy) and
   computationally cheap (gradients/optimizer state scale with trainable
   parameters only)

3. **Visualized ResNet-18 filters confirm the theory**: early layers detect
   oriented edges and color-opponent blobs, not object parts — exactly the
   generic features transfer learning depends on

4. **The from-scratch NumPy CNN generalizes beyond MNIST** to 3-channel
   CIFAR-10 images without modification to the underlying Conv2D/MaxPool2D
   layers, confirming those implementations are correctly general, not
   accidentally tuned to grayscale digits

5. **PyTorch and NumPy compute the same mathematical operation** — the
   speedup PyTorch provides is implementation efficiency (vectorized/compiled
   kernels), not a different algorithm

6. **CIFAR-10 is meaningfully harder than MNIST**: misclassifications
   concentrate on visually similar classes, unlike MNIST's near-perfectly
   separable digit classes


<a name="further-reading-8"></a>
### Further Reading

- PyTorch documentation: `torch.nn.Conv2d`, `torchvision.models`, transfer learning tutorial
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). "Deep Learning", Chapter 9 (Convolutional Networks)
- Stanford CS231n: assignment 2 (convolutional networks) and transfer-learning notes (cs231n.github.io)
- He, K., et al. (2015). "Deep Residual Learning for Image Recognition" (ResNet)
- Lesson 9a: CNN Theory (convolution, backprop, pooling derivations)
